# Робота з індексом стану рослинності

Тут реалізовані необхідні дії з даними, які було з сайту наукового центру NESDIS STAR

Створити віртуальне середовище (venv) в якому будуть встановлені всі необхідні бібліотеки та налаштування для даної лабораторної роботи

Ініціалізуємо необхідні бібліотеки

In [ ]:
from urllib.request import urlopen
import time
import os
import re
import glob
import pandas as pd
import numpy as np

Для кожної з адміністративних одиниць України завантажити (urllib) тестові структуровані файли, що містять значення VHI-індексу. При зберіганні файлу, до його імені потрібно додати дату та час завантаження. Передбачити повторні запуски скрипту, реалізувати механізм запобігання повторного довантаження та колізії даних.

In [ ]:
def file_already_exists(province_id):
    pattern = os.path.join("data",f"vhi_province_{province_id}_1981_2026_Mean_*.csv")
    files = glob.glob(pattern)
    return len(files) > 0

def download_file():
    parsed_url = "https://www.star.nesdis.noaa.gov/smcd/emb/vci/VH/get_TS_admin.php?country=UKR&provinceID={province_id}&year1=1981&year2=2026&type=Mean"
    os.makedirs("data", exist_ok=True)
    province_ids = list(range(1,28))

    for province_id in province_ids:
        if file_already_exists(province_id):
            continue

        url = parsed_url.format(province_id=province_id)

        current_time = time.strftime("%d-%m-%Y_%H-%M-%S")
        destination = os.path.join("data",f"vhi_province_{province_id}_1981_2026_Mean_{current_time}.csv")

        response = urlopen(url)
        text = response.read().decode("utf-8")

        clean_text = re.sub(r"<.*?>", "", text)

        with open(destination, "w", encoding="utf-8") as f:
            f.write(clean_text)

download_file()

Зчитати завантажені текстові файли у pandas dataframe. Здійснити data cleaning: прибрати зайві стовпці, заповнити пропуски, видалити зайвий текст тощо. Додати стовпчики з назвою та індексом області.

In [ ]:
def csv_to_df():
    files = glob.glob("data/*.csv")

    province_names = {
    1: "Cherkasy",
    2: "Chernihiv",
    3: "Chernivtsi",
    4: "Crimea",
    5: "Dnipropetrovs'k",
    6: "Donets'k",
    7: "Ivano-Frankivs'k",
    8: "Kharkiv",
    9: "Kherson",
    10: "Khmel'nyts'kyy",
    11: "Kiev",
    12: "Kiev City",
    13: "Kirovohrad",
    14: "Luhans'k",
    15: "L'viv",
    16: "Mykolayiv",
    17: "Odessa",
    18: "Poltava",
    19: "Rivne",
    20: "Sevastopol'",
    21: "Sumy",
    22: "Ternopil'",
    23: "Transcarpathia",
    24: "Vinnytsya",
    25: "Volyn",
    26: "Zaporizhzhya",
    27: "Zhytomyr"
}

    dfs = []

    for file in files:
        df = pd.read_csv(
            file,
            skiprows=2,
            header=None,
            names=["year", "week", "SMN", "SMT", "VCI", "TCI", "VHI"],
            usecols=[0, 1, 2, 3, 4, 5, 6],
            skipinitialspace=True
        )

        df.drop(columns=["SMN", "SMT", "VCI", "TCI"], inplace=True)

        df.replace(-1, np.nan, inplace=True)
        df.dropna(inplace=True)

        province_id = int(re.search(r'vhi_province_(\d+)', file).group(1))

        df["province_id"] = province_id
        df["province_name"] = province_names[province_id]

        dfs.append(df)

    return pd.concat(dfs, ignore_index=True)

vhi_df = csv_to_df()
vhi_df

Реалізувати процедуру зміни індексів: в завантажених з NOAA даних області індексуються за англійською абеткою (Province 1 - Cherkasy), потрібно замінити індекси так, щоб області індексувалася за українською абеткою (1 область - Вінницька).

In [ ]:
def change_indexes(df):
    changer_dict = {
        1: 25,
        2: 27,
        3: 26,
        4: 1,
        5: 4,
        6: 5,
        7: 9,
        8: 22,
        9: 23,
        10: 24,
        11: 11,
        12: 10,
        13: 12,
        14: 13,
        15: 14,
        16: 15,
        17: 16,
        18: 17,
        19: 18,
        20: 19,
        21: 20,
        22: 21,
        23: 7,
        24: 2,
        25: 3,
        26: 8,
        27: 6
    }

    df["province_id"] = df["province_id"].map(changer_dict)

change_indexes(vhi_df)
vhi_df

Реалізувати процедури для формування вибірок:

Ряд VHI для області за вказаний рік

In [ ]:
def filter_province_year(df):
    year = int(input("Введіть рік для виборки: "))
    province = int(input("Введіть індекс області для виборки: "))

    return df[(df["province_id"] == province) & (df["year"] == year)]

filter_province_year(vhi_df)

Ряд VHI за вказаний діапазон років для вказаних областей

In [ ]:
def filter_provinces_years(df):
    years = list(map(int, input("Введіть роки для виборки у форматі (yyyy yyyy yyyy): ").split()))
    provinces = list(map(int, input("Введіть індекси областей для виборки у форматі (ii ii ii): ").split()))

    return df[(df["province_id"].isin(provinces)) & (df["year"].isin(years))]

filter_provinces_years(vhi_df)

Пошук екстремумів (min та max) для вказаних областей та років, середнього, медіани

In [ ]:
def filter_stats_provinces_years(df):
    years = list(map(int, input("Введіть роки для виборки у форматі (yyyy yyyy yyyy): ").split()))
    provinces = list(map(int, input("Введіть індекси областей для виборки у форматі (ii ii ii): ").split()))

    df_filter_py = df[(df["province_id"].isin(provinces)) & (df["year"].isin(years))]
    stats = df_filter_py.groupby("province_name")["VHI"].agg(["min", "max", "mean", "median"])

    return stats


filter_stats_provinces_years(vhi_df)